# DSE 200 — Day 3 Lab, part 2
## Feature Engineering and Class Imbalance

**Agent on: explain every cell.**

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while we talk. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost. Every section rebuilds its own data.

In [ ]:
!uv pip install --system -q imbalanced-learn

In [ ]:
# ▶ SETUP — run first. Finds the data and defines check_read().
import os, textwrap, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

def find(name):
    """Return a local path for `name`, downloading it from the course repo if needed."""
    for p in [name, "/content/" + name, "data/" + name, "../" + name]:
        if os.path.exists(p):
            return p
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/" + name, name)
        print("downloaded", name, "from the course repo")
        return name
    except Exception as e:
        raise FileNotFoundError(f"{name} not found -- drag it into the Files pane (folder icon, left)") from e

TEMPS = find("hourly_temperatures_august.csv")
HOUSES = find("real_estate_san_diego.csv")
TRAFFIC = find("i5_traffic_data.csv")
LISTINGS = find("airbnb_sd_listings.csv")
CUSTOMERS = find("customers.csv")

FEATURES = ["n_orders", "total_spend", "avg_order_value",
            "days_since_last_order", "tenure_days", "n_categories"]

def clean(d):
    """The cleaning decisions from 6.7, so the imbalance sections start from the same table as this morning."""
    c = d.dropna(subset=FEATURES).copy()
    c = c[(c.total_spend >= 0) & (c.n_orders > 0) & (c.tenure_days <= 3650)].copy()
    c["avg_order_value"] = (c.total_spend / c.n_orders).round(2)
    return c

_READ = {
"8.2": ("Worse. Much worse: R² about -1.7 on the last nine days.",
        "Degree 2: -0.01. Degree 5: -0.18. Degree 7: -1.74.\n"
        "The test set is the FUTURE -- time indexes the model never saw.\n"
        "A polynomial bends to fit the training days, then shoots off past the end of them.\n"
        "More flexibility made the forecast worse. The problem was never the curve's shape;\n"
        "it was that 'hours since August 1' is the wrong column. The day repeats. Say so."),
"8.3": ("About 0.14. Location sets the price and the model still cannot see it.",
        "A linear model on latitude and longitude can only tilt a flat plane across the map:\n"
        "'pricier to the north-west', say. Here price PEAKS at two points and falls away in\n"
        "every direction from each. No tilted plane fits two hills.\n"
        "Give it 'miles to the nearest centre' and the same model gets 0.94.\n"
        "The model did not get smarter. The column did."),
"8.4": ("About 0.17. Barely better.",
        "hour + is_rush: R² 0.17. The column is right on weekdays and wrong on weekends:\n"
        "Saturday at 8 am is quiet, but is_rush says 'rush'. A third of the training rows\n"
        "are lying to the model. The fix is not a fancier model, it is a better column:\n"
        "rush hour on WEEKDAYS. Look at the scatter again -- it was telling you."),
"8.5": ("'dorm', then 'student'.",
        "Most negative: dorm, student, room, private bedroom, tiny, shared, hostel.\n"
        "Most positive: estate, oceanfront, units, villa, hot (as in tub), overlooking.\n"
        "Every one is a fact about the listing the host chose to advertise:\n"
        "a shared room near campus, or a house on the water. The words ARE the features.\n"
        "(Last year's answer key listed 'luxury', 'cozy', 'budget' -- the model never said that.)"),
"9.2": ("Accuracy 0.99. Cat recall 0.50.",
        "Puppy is positive. TP 98, FP 1 (the missed cat), FN 0, TN 1.\n"
        "puppy  precision 98/99 = 0.99   recall 98/98 = 1.00   F1 0.99\n"
        "cat    precision  1/1  = 1.00   recall  1/2  = 0.50   F1 0.67\n"
        "macro F1 = (0.99 + 0.67) / 2 = 0.83\n"
        "Accuracy averages over ROWS, so 98 puppies drown 2 cats.\n"
        "Macro F1 averages over CLASSES, so the cat gets half the vote."),
"9.5": ("It jumps from about 0.57 to about 0.86. And it is a lie.",
        "SMOTE makes new returners by drawing lines between real ones.\n"
        "Resample BEFORE the split and the test set is full of points that sit on a line\n"
        "between two training points. The forest has seen both ends.\n"
        "It is not predicting; it is recognising. Same leak as a row in both halves (7.1).\n"
        "Rule: split first. Resample the training half only. Never touch the test set."),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print("data found, check_read() and clean() ready.")

# Feature Engineering

*Slides: Feature Engineering · sections 8.1–8.5*

## 8.1 The Column Is Where the Knowledge Goes

A linear model can only add up the columns you hand it, each times a number. It cannot discover that 11 pm is next to midnight, or that two neighbourhoods are expensive. **You know those things. A feature is how you tell the model.**

Four cases, same recipe each time: a baseline that fails, the reason it fails, one new column, the same model again.

## 8.2 Time: the Day Repeats

Hourly temperatures for August. Train on the first 70% of the month, test on the last 30%. **Not a random split:** with time, a random split lets the model peek at tomorrow to predict today.

In [ ]:
# ▶ SETUP 8.2 — from cold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

temps = pd.read_csv(TEMPS, parse_dates=["timestamp"])
temps["t"] = range(len(temps))                       # hours since Aug 1, 00:00
cut = int(len(temps) * 0.7)
train, test = temps.iloc[:cut].copy(), temps.iloc[cut:].copy()

def score(cols, train=train, test=test, target="temperature_f"):
    """Fit LinearRegression on train[cols], return (R², MAE) on test."""
    m = LinearRegression().fit(train[cols], train[target])
    p = m.predict(test[cols])
    return round(r2_score(test[target], p), 3), round(mean_absolute_error(test[target], p), 2)

print(len(train), "train hours  ", len(test), "test hours")
print("test starts", test.timestamp.iloc[0])

In [ ]:
# ▶ DEMO -- look before you model
temps.iloc[:24 * 5].plot(x="timestamp", y="temperature_f", legend=False, figsize=(10, 3))
plt.title("five days: up every afternoon, down every night"); plt.ylabel("°F"); plt.show()

In [ ]:
# ▶ DEMO -- the baseline: hours since the start of the month
print("time index  R², MAE:", score(["t"]))

**R² ≈ 0.** A straight line through a month of ups and downs is flat. The model learned the average, nothing more.

In [ ]:
# 🔍 READ 8.2 — give the line more freedom: polynomial features of t, degree 2, 3, 5, 7.
#
#     PolynomialFeatures(degree=7) on t, then the same LinearRegression.
#
# Better or worse than R² ≈ 0 on the last nine days? By roughly how much? Commit first.

prediction = "???"
check_read("8.2", prediction)

In [ ]:
# ▶ DEMO -- see it
from sklearn.preprocessing import PolynomialFeatures
for d in [2, 3, 5, 7]:
    poly = PolynomialFeatures(d).fit(train[["t"]])
    m = LinearRegression().fit(poly.transform(train[["t"]]), train.temperature_f)
    p = m.predict(poly.transform(test[["t"]]))
    print(f"degree {d}   R² {r2_score(test.temperature_f, p):7.3f}")

In [ ]:
# ▶ DEMO -- closer: the hour of the day, as a number
for df_ in (train, test):
    df_["hour"] = df_.timestamp.dt.hour
print("hour as a number  R², MAE:", score(["hour"]))

**0.65. Better, and still wrong in a specific way.** To the model, hour 23 and hour 0 are 23 apart. On the clock they are neighbours. Put the hour on a circle: `sin` and `cos` of the angle give every hour a position where 23 sits next to 0.

```
angle = 2π · hour / 24        hour_sin = sin(angle)        hour_cos = cos(angle)
```

In [ ]:
# ✍️ YOU TRY 8.2
# 1. Add hour_sin and hour_cos to BOTH train and test (the loop above shows how).
# 2. r2_cyclic -- the R² from score(["hour_sin", "hour_cos"]).

# TODO

r2_cyclic = 0.0   # TODO
print("sin/cos  R²:", r2_cyclic)

In [ ]:
# ✅ CHECKPOINT 8.2
assert {"hour_sin", "hour_cos"} <= set(train.columns) and {"hour_sin", "hour_cos"} <= set(test.columns)
assert abs(train.hour_sin.pow(2) + train.hour_cos.pow(2) - 1).max() < 1e-9, "sin² + cos² should be 1"
assert r2_cyclic > 0.85, r2_cyclic
print(f"Correct. Two columns took R² from 0 to {r2_cyclic}. Same model, same data.")

In [ ]:
# ▶ DEMO -- the other honest encoding: one column per hour
onehot = lambda d: pd.get_dummies(d.timestamp.dt.hour, prefix="h").astype(int)
tr, te = pd.concat([train, onehot(train)], axis=1), pd.concat([test, onehot(test)], axis=1)
print("one-hot hour  R², MAE:", score([f"h_{h}" for h in range(24)], tr, te))

**One-hot does a little better here: 24 columns let each hour have its own level.** Sin/cos does nearly as well with two, and says something one-hot cannot: that neighbouring hours are similar. With 744 rows either works. With a day-of-year cycle and 3 years of data, 365 one-hot columns is a lot of coefficients to learn from three examples each.

In [ ]:
# ▶ DEMO -- lag features: what the temperature WAS
lagged = temps.copy()
angle = 2 * np.pi * lagged.timestamp.dt.hour / 24
lagged["hour_sin"], lagged["hour_cos"] = np.sin(angle), np.cos(angle)
for h in [1, 3, 24]:
    lagged[f"lag_{h}h"] = lagged.temperature_f.shift(h)
lagged = lagged.dropna()                      # the first 24 hours have no 24-hour lag
c2 = int(len(lagged) * 0.7)
tr, te = lagged.iloc[:c2], lagged.iloc[c2:]

print("sin/cos + lag_24h          ", score(["hour_sin", "hour_cos", "lag_24h"], tr, te))
print("sin/cos + lag_1h,3h,24h    ", score(["hour_sin", "hour_cos", "lag_1h", "lag_3h", "lag_24h"], tr, te))

**The best number on this page comes with a question you must ask every time: would I have this column when I need the prediction?**

`lag_1h` means "I know the temperature an hour ago". Fine for a thermostat deciding the next hour. Useless for a forecast of tomorrow afternoon, made this morning: that lag doesn't exist yet. Using it anyway is the leak from this morning in a new costume. `lag_24h` is the honest one for a day-ahead forecast.

## 8.3 Location: Two Hills

San Diego house sales: latitude, longitude, bedrooms, square feet, price. Prices are highest near **downtown** and near **UCSD**.

In [ ]:
# ▶ SETUP 8.3 — from cold
from sklearn.model_selection import train_test_split
houses = pd.read_csv(HOUSES)
DOWNTOWN = (32.715736, -117.161087)
UCSD = (32.8812, -117.2344)
h_train, h_test = train_test_split(houses, test_size=0.3, random_state=42)

def score_h(cols):
    """Fit LinearRegression on h_train[cols], return (R², MAE in dollars) on h_test."""
    m = LinearRegression().fit(h_train[cols], h_train.price)
    p = m.predict(h_test[cols])
    return round(r2_score(h_test.price, p), 3), round(mean_absolute_error(h_test.price, p))

houses.describe().round(2)

In [ ]:
# ▶ DEMO -- the map
plt.figure(figsize=(9, 4))
plt.scatter(houses.longitude, houses.latitude, c=houses.price / 1e6, s=15, cmap="viridis")
plt.colorbar(label="price ($M)")
for (lat, lon), name in [(DOWNTOWN, "downtown"), (UCSD, "UCSD")]:
    plt.scatter(lon, lat, marker="*", s=300, c="red", edgecolors="black")
    plt.annotate(name, (lon, lat), xytext=(6, 6), textcoords="offset points")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.title("two expensive centres, cheaper in every direction away from them"); plt.show()

In [ ]:
# 🔍 READ 8.3 — the obvious model: price from latitude and longitude.
#
#     score_h(["latitude", "longitude"])
#
# Location clearly sets the price on that map. What R² do you expect? Commit first.

prediction = "???"
check_read("8.3", prediction)

In [ ]:
# ▶ DEMO
print("lat/lon  R², MAE:", score_h(["latitude", "longitude"]))

**Distance is the feature.** Miles between two points on a sphere is the haversine formula. It is written for you, and it works on whole columns at once.

In [ ]:
# ▶ DEMO -- distance in miles, vectorised: no loop, no extra library
def haversine_miles(lat, lon, lat0, lon0):
    """Great-circle distance in miles from (lat, lon) -- scalars or columns -- to one point."""
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, lat0, lon0))
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))

print("downtown to UCSD:", round(haversine_miles(*DOWNTOWN, *UCSD), 1), "miles")   # about 12

In [ ]:
# ✍️ YOU TRY 8.3
# 1. Add three columns to BOTH h_train and h_test:
#      d_downtown -- miles to DOWNTOWN
#      d_ucsd     -- miles to UCSD
#      d_min      -- miles to whichever is nearer (np.minimum)
# 2. r2_dist -- the R² from score_h(["d_min"])

# TODO

r2_dist = 0.0   # TODO
print("nearest-centre distance  R²:", r2_dist)

In [ ]:
# ✅ CHECKPOINT 8.3
assert {"d_downtown", "d_ucsd", "d_min"} <= set(h_test.columns)
assert (h_train.d_min <= h_train.d_downtown).all() and (h_train.d_min <= h_train.d_ucsd).all()
assert 0 < h_train.d_downtown.min() and h_train.d_downtown.max() < 80, "miles, not km or degrees?"
assert r2_dist > 0.9, r2_dist
print(f"Correct. One column: R² 0.14 -> {r2_dist}.")

In [ ]:
# ▶ DEMO -- every version, side by side (rebuilds the distance columns, so it runs from cold)
for df_ in (h_train, h_test):
    df_["d_downtown"] = haversine_miles(df_.latitude, df_.longitude, *DOWNTOWN)
    df_["d_ucsd"] = haversine_miles(df_.latitude, df_.longitude, *UCSD)
    df_["d_min"] = np.minimum(df_.d_downtown, df_.d_ucsd)

for cols in [["latitude", "longitude"], ["d_downtown"], ["d_ucsd"],
             ["d_downtown", "d_ucsd"], ["d_min"], ["d_min", "sqft", "bedrooms"]]:
    r2, mae = score_h(cols)
    print(f"{' + '.join(cols):28}  R² {r2:6.3f}   MAE ${mae:>9,}")

**Read the middle rows.** Each single distance knows about one hill. Both distances together beat either one alone, but the model can only *add* them, so a house near UCSD is still penalised for being far from downtown. `d_min` says what's actually true: "near either one is good". That rule came from you looking at the map, not from the model.

## 8.4 Traffic: Hours That Belong Together

Hourly vehicle counts on I-5, four weeks starting Monday, August 5. Train on the first 70%, test on the rest, in time order again.

In [ ]:
# ▶ SETUP 8.4 — from cold
traffic = pd.read_csv(TRAFFIC, parse_dates=["timestamp"])
traffic["hour"] = traffic.timestamp.dt.hour
traffic["weekend"] = (traffic.timestamp.dt.day_of_week >= 5).astype(int)
cut = int(len(traffic) * 0.7)
t_train, t_test = traffic.iloc[:cut].copy(), traffic.iloc[cut:].copy()

def score_t(cols, train=None, test=None):
    """Fit LinearRegression on train[cols], return (R², MAE in vehicles) on test."""
    train = t_train if train is None else train
    test = t_test if test is None else test
    m = LinearRegression().fit(train[cols], train.vehicle_volume)
    p = m.predict(test[cols])
    return round(r2_score(test.vehicle_volume, p), 3), round(mean_absolute_error(test.vehicle_volume, p))

print(len(t_train), "train hours  ", len(t_test), "test hours   test starts", t_test.timestamp.iloc[0])

In [ ]:
# ▶ DEMO -- look first
for wk, name in [(0, "weekday"), (1, "weekend")]:
    d = traffic[traffic.weekend == wk]
    plt.scatter(d.hour, d.vehicle_volume, s=10, alpha=0.5, label=name)
plt.xlabel("hour of day"); plt.ylabel("vehicles per hour"); plt.legend()
plt.title("two humps on weekdays, one lazy hill on weekends"); plt.show()
print("hour as a number  R², MAE:", score_t(["hour"]))

**Hour as a number fails for the same reason as 8.2,** and sin/cos won't save it: a cycle has one peak, this has two. Domain knowledge says rush hour is 7 to 9 and 4 to 6.

In [ ]:
# 🔍 READ 8.4 — add one column: is_rush = 1 for hours 7, 8, 9, 16, 17, 18.
#
#     score_t(["hour", "is_rush"])
#
# R² from about 0 to what? Commit first.

prediction = "???"
check_read("8.4", prediction)

In [ ]:
# ▶ DEMO
for df_ in (t_train, t_test):
    df_["is_rush"] = df_.hour.isin([7, 8, 9, 16, 17, 18]).astype(int)
print("hour + is_rush  R², MAE:", score_t(["hour", "is_rush"]))

In [ ]:
# ✍️ YOU TRY 8.4
# 1. Add rush_weekday to BOTH t_train and t_test: 1 only for rush hours on a weekday.
# 2. r2_rush -- the R² from score_t(["rush_weekday", "weekend"])

# TODO

r2_rush = 0.0   # TODO
print("weekday rush + weekend  R²:", r2_rush)

In [ ]:
# ✅ CHECKPOINT 8.4
assert "rush_weekday" in t_test.columns
assert t_train.loc[t_train.weekend == 1, "rush_weekday"].sum() == 0, "no rush hour on weekends"
assert r2_rush > 0.5, r2_rush
print(f"Correct. One honest column took R² from 0.17 to {r2_rush}.")

**Can the data find the categories itself?** There are 48 kinds of hour: 24 on weekdays and 24 on weekends. Average each one's volume **on the training weeks only**, then let KMeans sort those 48 averages into 4 groups. Every row gets its group as a one-hot feature.

In [ ]:
# ▶ DEMO -- clustering the 48 (weekend, hour) slots by their TRAINING average
from sklearn.cluster import KMeans
slot_mean = t_train.groupby(["weekend", "hour"]).vehicle_volume.mean()
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(slot_mean.values.reshape(-1, 1))
slot_group = dict(zip(slot_mean.index, km.labels_))

for df_ in (t_train, t_test):
    df_["slot"] = [slot_group[k] for k in zip(df_.weekend, df_.hour)]
onehot_slot = lambda d: pd.concat([d, pd.get_dummies(d.slot, prefix="slot").astype(int)], axis=1)
tr, te = onehot_slot(t_train), onehot_slot(t_test)
print("4 slot groups  R², MAE:", score_t([f"slot_{k}" for k in range(4)], tr, te))

print("\nwhich group each hour landed in (columns: weekday 0 / weekend 1):")
pd.Series(slot_group).unstack(0)

**0.88: better than any rule we wrote by hand.** Read the table: one group is night, one is daytime, one is rush hour, and weekday 5 pm gets a group of its own. Weekend mornings land with night and weekend afternoons with weekday daytime, and nobody told it that.

**Why this is allowed, and last year's version wasn't.** The 2025 notebook clustered rows on `vehicle_volume` *itself*, the column it then predicted, so every test row's cluster was chosen by looking at its answer. Here each slot's average comes from training weeks only, and at prediction time you know the hour and the day. Same rule as the lags in 8.2: **would I have it when I need the prediction?**

## 8.5 Text: Words Are Columns Too

San Diego Airbnb listings (Inside Airbnb, December 2021, CC BY 4.0). Can the listing's name predict its price?

In [ ]:
# ▶ SETUP 8.5 — from cold
listings = pd.read_csv(LISTINGS)
listings = listings[listings.price > 0].dropna(subset=["name"]).copy()
listings["log_price"] = np.log(listings.price)
a_train, a_test = train_test_split(listings, test_size=0.3, random_state=42)

def score_a(X_train, X_test, model=None):
    """Fit on log price, return (R² on log price, median-ish MAE in dollars) on the test listings."""
    model = model or LinearRegression()
    p = model.fit(X_train, a_train.log_price).predict(X_test)
    return round(r2_score(a_test.log_price, p), 3), round(mean_absolute_error(a_test.price, np.exp(p)))

print(f"{len(listings):,} listings   mean ${listings.price.mean():.0f}   median ${listings.price.median():.0f}   max ${listings.price.max():,.0f}")
listings[["name", "room_type", "price"]].sample(6, random_state=1)

**Mean $290, median $176, max $10,000: a long right tail (6.3).** Predict `log(price)` so a $10,000 villa doesn't outvote a thousand ordinary listings, then turn predictions back into dollars with `exp`.

In [ ]:
# ▶ DEMO -- baseline: room type only
room = lambda d: pd.get_dummies(d.room_type).reindex(columns=sorted(listings.room_type.unique()), fill_value=0).astype(int)
print("room type  R², MAE:", score_a(room(a_train), room(a_test)))

In [ ]:
# ▶ DEMO -- how text becomes columns: one column per word, one row per listing
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
toy = ["ocean view villa", "cozy room near ocean", "private room"]
for vec in (CountVectorizer(), TfidfVectorizer()):
    m = vec.fit_transform(toy)
    print(type(vec).__name__); print(pd.DataFrame(m.toarray().round(2), columns=vec.get_feature_names_out(), index=toy), "\n")

**Counts treat every word alike. TF-IDF down-weights words that appear everywhere** ("room", "ocean" here) and up-weights rare ones ("villa"). With real listings, `stop_words="english"` drops "the", "and", "in", and `ngram_range=(1, 2)` adds word pairs like "hot tub", "la jolla".

In [ ]:
# 🔍 READ 8.5 — fit TF-IDF on the training names and a linear model on log price.
#
# Which word gets the MOST NEGATIVE coefficient -- the word that says "cheap"? Commit first.

prediction = "???"
check_read("8.5", prediction)

In [ ]:
# ✍️ YOU TRY 8.5
# 1. tfidf -- TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1, 2)),
#             FIT ON a_train.name ONLY. Then .transform both a_train.name and a_test.name.
# 2. r2_text -- the R² from score_a(X_train_text, X_test_text, Ridge(alpha=1.0))
#    Ridge is LinearRegression with a brake on huge coefficients; 500 overlapping word columns need one.
from sklearn.linear_model import Ridge

tfidf = None    # TODO
r2_text = 0.0   # TODO
print("listing name  R²:", r2_text)

In [ ]:
# ✅ CHECKPOINT 8.5
assert tfidf is not None and len(tfidf.get_feature_names_out()) == 500
assert "la jolla" in tfidf.vocabulary_, "word pairs missing: ngram_range=(1, 2)?"
assert "the" not in tfidf.vocabulary_, "stop words still in"
assert r2_text > 0.38, r2_text
print(f"Correct. The name alone explains {r2_text:.0%} of log price; room type explained 22%.")

In [ ]:
# ▶ DEMO -- what the model learned, and both together (rebuilds tfidf, so it runs from cold)
tfidf = TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1, 2)).fit(a_train.name)
X_train_text, X_test_text = tfidf.transform(a_train.name), tfidf.transform(a_test.name)
ridge = Ridge(alpha=1.0).fit(X_train_text, a_train.log_price)
coef = pd.Series(ridge.coef_, tfidf.get_feature_names_out()).sort_values()
print("cheapest words:", ", ".join(coef.head(8).index))
print("priciest words:", ", ".join(coef.tail(8).index[::-1]))

from scipy.sparse import hstack, csr_matrix
both = lambda X, d: hstack([X, csr_matrix(room(d).values)])
print("\nname + room type  R², MAE:", score_a(both(X_train_text, a_train), both(X_test_text, a_test), Ridge(alpha=1.0)))

In [ ]:
# ▶ DEMO -- a number hiding in the text
beds = listings.name.str.lower().str.extract(r"(\d+)\s*(?:br|bd|bdrm|bed|bedroom)s?\b")[0].astype(float)
beds = beds.where(beds < 15)
print(f"{beds.notna().mean():.0%} of names state a bedroom count")
listings.assign(beds_in_name=beds).groupby("beds_in_name").price.median().head(8).round()

**Median price climbs with every bedroom the host typed into the title.** That's a numeric column nobody gave you, sitting in free text. One regular expression turns it into a feature.

**On your own data:** a date column hides an hour, a weekday, a month, a "days since". An address hides a distance. A number column might really be a category (8.4). A free-text field hides words, a length, and sometimes a number.

# Class Imbalance

*Slides D03.07 · sections 9.1–9.5 · bring your missed-returner count from 7.4*

## 9.1 The Problem You Already Have

This morning: logistic regression, 0.823 accuracy, **255 of 419 returners missed**. 22% of customers return. A model that always says "no" scores 0.777 without learning anything.

In [ ]:
# ▶ SETUP 9.1 — from cold: same split as 7.1
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report, precision_score,
                             recall_score, f1_score, roc_auc_score, RocCurveDisplay)
c = clean(pd.read_csv(CUSTOMERS))
X, y = c[FEATURES], c["returned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
tn, fp, fn, tp = confusion_matrix(y_test, logreg.predict(X_test)).ravel()
print(f"positives: {y_train.mean():.1%} of train   accuracy {logreg.score(X_test, y_test):.3f}")
print(f"returners in test {tp + fn}: caught {tp}, missed {fn}")

## 9.2 Metrics That Notice

Precision: of the rows we called positive, how many were. Recall: of the positives, how many we caught. F1 balances the two. **Macro F1** computes F1 for each class and averages them, so the small class gets an equal vote.

In [ ]:
# 🔍 READ 9.2 — the slides' example. Test set: 98 puppies, 2 cats. Puppy is the positive class.
# The model says puppy 99 times and cat once. The one it calls a cat IS a cat.
#
# Accuracy? Recall on cats? Macro F1? Commit to all three.

prediction = "???"
check_read("9.2", prediction)

In [ ]:
# ▶ DEMO -- check it with sklearn, not by hand
actual    = np.array(["puppy"] * 98 + ["cat"] * 2)
predicted = np.array(["puppy"] * 98 + ["puppy", "cat"])
print(classification_report(actual, predicted, digits=2))

In [ ]:
# ▶ DEMO -- the same report on our customers
print(classification_report(y_test, logreg.predict(X_test), digits=3))

**Two lines, two stories.** Class 0: precision 0.84, recall 0.95, it's easy to be right about people who don't return. Class 1: recall 0.39. Accuracy is the weighted average, so it inherits class 0's good news.

**AUC asks a different question:** pick one returner and one non-returner at random. How often does the model give the returner the higher probability? 0.5 is a coin, 1.0 is perfect. It ignores the 0.5 cut-off completely, so it measures how well the model **ranks**, not where it draws the line.

In [ ]:
# ▶ DEMO
proba = logreg.predict_proba(X_test)[:, 1]
print("AUC", round(roc_auc_score(y_test, proba), 3))
RocCurveDisplay.from_predictions(y_test, proba)
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.title("every possible threshold at once"); plt.show()

## 9.3 Rebalancing

Four ways, all from the slides:

| Slides | Here | What it does |
| --- | --- | --- |
| Remove some dogs | **undersample** | drop non-returners from TRAIN until the classes match |
| Add more cats | **oversample** | repeat returners in TRAIN until they match |
| Penalise mistakes on cats | **class weights** | `class_weight="balanced"`: each class's errors cost `n / (2 · n_class)` |
| Synthetic data | **SMOTE** | invent new returners between real ones, in TRAIN |

Every one of them touches **training rows only**. The test set stays what the world looks like.

In [ ]:
# ▶ DEMO -- the weight formula from the slides, on our data
from sklearn.utils.class_weight import compute_class_weight
n, n0, n1 = len(y_train), (y_train == 0).sum(), (y_train == 1).sum()
print("by hand :", round(n / (2 * n0), 3), round(n / (2 * n1), 3))
print("sklearn :", compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train).round(3))
print(f"so missing one returner costs as much as {n0 / n1:.1f} false alarms")

In [ ]:
# ▶ DEMO -- one helper, so every strategy is scored the same way
def report(model, X_fit=None, y_fit=None, threshold=0.5):
    """Fit on (X_fit, y_fit) -- default the plain training set -- and score on the untouched test set."""
    X_fit = X_train if X_fit is None else X_fit
    y_fit = y_train if y_fit is None else y_fit
    p = model.fit(X_fit, y_fit).predict_proba(X_test)[:, 1]
    pred = (p >= threshold).astype(int)
    return {"precision": round(precision_score(y_test, pred), 3),
            "recall": round(recall_score(y_test, pred), 3),
            "macro_f1": round(f1_score(y_test, pred, average="macro"), 3),
            "auc": round(roc_auc_score(y_test, p), 3),
            "missed": int(((y_test == 1) & (pred == 0)).sum())}

pd.DataFrame({"plain": report(LogisticRegression(max_iter=1000)),
              "balanced": report(LogisticRegression(max_iter=1000, class_weight="balanced"))}).T

In [ ]:
# ✍️ YOU TRY 9.3
# results -- a dict: strategy name -> report(...) for "undersample", "oversample", "smote".
#   train = pd.concat([X_train, y_train], axis=1) gives you one table to sample from.
#   undersample: all returners + an equal-size .sample(random_state=42) of non-returners
#   oversample:  all non-returners + returners .sample(n=..., replace=True, random_state=42)
#   smote:       SMOTE(random_state=42).fit_resample(X_train, y_train)
#   Each one: report(LogisticRegression(max_iter=1000), X_fit, y_fit)
from imblearn.over_sampling import SMOTE

results = {
    "plain": report(LogisticRegression(max_iter=1000)),
    "balanced": report(LogisticRegression(max_iter=1000, class_weight="balanced")),
}
# TODO: add "undersample", "oversample", "smote"

pd.DataFrame(results).T

In [ ]:
# ✅ CHECKPOINT 9.3
assert set(results) == {"plain", "balanced", "undersample", "oversample", "smote"}, sorted(results)
for k in ["balanced", "undersample", "oversample", "smote"]:
    assert results[k]["recall"] > 0.75, (k, results[k])        # all catch most returners now
    assert results[k]["missed"] < 100, (k, results[k])
aucs = [r["auc"] for r in results.values()]
assert max(aucs) - min(aucs) < 0.03, aucs
print("Correct.\n")
print(f"missed returners: {results['plain']['missed']} -> {results['balanced']['missed']}")
print(f"AUC across all five: {min(aucs)} to {max(aucs)}")

**Look at the AUC column before you celebrate.** Every strategy rescued about 190 returners, and AUC barely moved. The model doesn't rank customers any better than before. **Rebalancing moved the line, not the model:** it now says "returner" at a lower probability, which catches more returners and raises more false alarms (look at precision).

If that's all it does, you can move the line yourself.

## 9.4 Move the Threshold

In [ ]:
# ▶ DEMO -- the plain model, five different lines
plain = LogisticRegression(max_iter=1000)
pd.DataFrame({t: report(plain, threshold=t) for t in [0.5, 0.4, 0.3, 0.22, 0.2]}).T.rename_axis("threshold")

**At 0.22 (the share of returners) the plain model matches `class_weight="balanced"` almost exactly.** Same caught, same false alarms, no retraining.

Which threshold is right isn't a data science question. It's this one: **what does a missed returner cost, and what does a false alarm cost?** A coupon to someone who wasn't coming back costs a few dollars. Missing a customer who was about to churn might cost a year of orders. Get those two numbers from whoever owns the decision, then pick the line.

## 9.5 The SMOTE Trap

In [ ]:
# 🔍 READ 9.5 — a random forest with SMOTE, done two ways.
#
#   RIGHT: split, then SMOTE the training half only           -> F1 on returners about 0.57
#   WRONG: SMOTE the whole table, then split                  -> F1 on returners = ???
#
# Higher, lower, or about the same? Commit first.

prediction = "???"
check_read("9.5", prediction)

In [ ]:
# ▶ DEMO -- see it
from sklearn.ensemble import RandomForestClassifier
rf = lambda: RandomForestClassifier(n_estimators=200, random_state=42)

m = rf().fit(*SMOTE(random_state=42).fit_resample(X_train, y_train))
print("RIGHT  F1 on returners:", round(f1_score(y_test, m.predict(X_test)), 3))

X_all, y_all = SMOTE(random_state=42).fit_resample(X, y)
a, b, ya, yb = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
m = rf().fit(a, ya)
print("WRONG  F1 on returners:", round(f1_score(yb, m.predict(b)), 3), " <- the leak, not the model")

## Wrap

You can now:

- Turn a cycle into two columns with sin and cos, and say why a polynomial of time fails out of range
- Ask of every lag: would I have this when I need the prediction?
- Turn coordinates into a distance, and turn "near either one" into a `minimum`
- Turn hours into categories by hand (rush hour, weekdays only) or with KMeans fit on training data
- Turn text into TF-IDF columns fit on training rows, and pull numbers out of it with a regex
- Read precision, recall, F1, and macro F1 per class, and say what accuracy averages over
- Read AUC as "how well does it rank", separate from the threshold
- Rebalance with weights, under- and oversampling, or SMOTE, on training rows only
- Show that rebalancing mostly moves the threshold, and move it yourself
- Spot the SMOTE leak

**On your own data:** HW3 Part 3 has you fit a classifier. If your positive class is under 30%, report recall and macro F1 next to accuracy, and say which threshold you'd use and why.